# Job quality research · second evidence pass
**14 November 2026 · offline CSV edition**

Research questions concern job attributes and worker outcomes. Discovery counts do not rank attributes, establish country health differences or estimate policy effects.

This pass screens the **87 original security/hours candidate publications**, not every expanded-search result. It includes **24 extracted estimates from 10 publications: 20 full-text table checks and 4 abstract checks**. This is a single AI-assisted pass; independent verification and formal bias assessment remain pending.

**Colab:** unzip `job_quality_v02_csv_files.zip` on your computer. Run the cells; when prompted, select its CSV files together. Locally, keep this notebook next to `data/`. All research data are ordinary readable CSVs; there is no embedded payload. No API key is required for the completed snapshot analysis.


## Combined relationship graph — reproducible Python

Run the next two code cells to **generate the single graph** and save PNG/SVG copies in `figures/`. They are self-contained and can run before uploading the research CSVs. All numerical inputs, transformations and source links are visible below. No embedded image or encoded dataset is required.

This graph includes the later seven-relationship follow-up. Its supplementary results are **not included in the original 24-estimate extraction count** elsewhere in this notebook. The two life-satisfaction papers supplement the System Graph snapshot; neither DOI was found in that snapshot. The salary–mortality values are abstract-checked, with confidence intervals unverified. The working-conditions physical-health estimate has a flagged abstract/Results discrepancy.

The diagram summarizes selected observational findings. Its colors are not risk-of-bias grades, and its numbers must not be pooled or used to rank attributes. Detailed provenance is in `combined_graph_sources.md` in the full package.


In [ ]:
# 1. Explicit research inputs. These are published estimates, not simulated observations.
import math

GRAPH_EVIDENCE = {
    "salary_mortality": {
        "rr": 1.59,
        "ci": None,  # No verified interval; never replace missing uncertainty with zero.
        "population": "Korean male civil servants, 1992–1996",
        "contrast": "lowest versus highest salary grade",
        "source": "https://pubmed.ncbi.nlm.nih.gov/10746109/",
        "locator": "Abstract; System Graph association 020zZ9sQBg",
    },
    "pay_life_satisfaction": {
        "beta_log_hourly_pay": 0.250,
        "robust_se": 0.010,
        "population": "UK Annual Population Survey; full-time workers aged 18–65",
        "source": "https://onlinelibrary.wiley.com/doi/10.1111/ecca.12516",
        "locator": "Table 1; primary-paper supplement, not matched in snapshot",
    },
    "conditions_physical": {
        "individual_hr_abstract": 1.32,
        "individual_hr_results_text": 1.31,  # Source discrepancy remains unresolved.
        "individual_ci": (1.13, 1.53),
        "work_unit_hr": 1.02,
        "work_unit_ci": (0.88, 1.19),
        "source": "https://www.sjweh.fi/show_abstract.php?abstract_id=3402&fullText=1",
        "locator": "Abstract and Results; Model 2; Graph seed OCkbXWwzaH",
    },
    "conditions_mental": {
        "individual_hr": 1.90,
        "individual_ci": (1.51, 2.40),
        "work_unit_hr": 1.63,
        "work_unit_ci": (1.31, 2.04),
        "source": "https://www.sjweh.fi/show_abstract.php?abstract_id=3402&fullText=1",
        "locator": "Results; Model 2; depression-related disability pension",
    },
    "hours_depression": {
        "main_or": 2.52,
        "main_ci": (1.12, 5.65),
        "sensitivity_or": 1.89,
        "sensitivity_ci": (0.92, 3.86),
        "source": "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0030719",
        "locator": "Table 2 / Results; prior v01 paper check; Graph qtGTRpRBv0",
    },
    "insecurity_depression": {
        "men_or": 2.47,
        "men_ci": (1.37, 4.48),
        "women_or": 1.48,
        "women_ci": (0.88, 2.50),
        "source": "https://pmc.ncbi.nlm.nih.gov/articles/PMC8949946/",
        "locator": "Tables 5–6, separate-indicator models; v02 E01 and E03",
    },
    "insecurity_life_satisfaction": {
        "adjusted_beta": -0.26,
        "se": 0.16,
        "contrast": "worried versus not worried, conditional on permanent employment",
        "source": "https://pmc.ncbi.nlm.nih.gov/articles/PMC5984641/",
        "locator": "Table 3 M2; initial life satisfaction; primary-paper supplement",
    },
}

# 2. Transparent transformations. The papers report coefficients and standard errors.
# Pay: a doubling changes log(hourly pay) by ln(2); scale the coefficient and SE.
pay = GRAPH_EVIDENCE["pay_life_satisfaction"]
pay_difference = pay["beta_log_hourly_pay"] * math.log(2)
pay_margin = 1.96 * pay["robust_se"] * math.log(2)
pay_ci = (pay_difference - pay_margin, pay_difference + pay_margin)

# Insecurity: normal-approximation CI from the rounded published coefficient and SE.
security = GRAPH_EVIDENCE["insecurity_life_satisfaction"]
security_ci = (
    security["adjusted_beta"] - 1.96 * security["se"],
    security["adjusted_beta"] + 1.96 * security["se"],
)

# Labels/colors summarize these selected findings, not the quality of all evidence.
salary = GRAPH_EVIDENCE["salary_mortality"]
physical = GRAPH_EVIDENCE["conditions_physical"]
mental = GRAPH_EVIDENCE["conditions_mental"]
hours = GRAPH_EVIDENCE["hours_depression"]
insecurity = GRAPH_EVIDENCE["insecurity_depression"]
graph_rows = [
    ("Physical health", "Lower salary grade, higher mortality",
     f"RR {salary['rr']:.2f} • confidence interval not verified", "gray", salary["source"]),
    ("Life satisfaction", "Higher hourly pay, higher life satisfaction†",
     f"UK: +{pay_difference:.2f} points per doubling of pay (approx. CI {pay_ci[0]:.2f}–{pay_ci[1]:.2f})",
     "teal", pay["source"]),
    ("Physical health", "Effort–reward imbalance: findings vary",
     f"Musculoskeletal disability: HR {physical['individual_hr_abstract']:.2f}* (individual) / {physical['work_unit_hr']:.2f} (work unit)",
     "amber", physical["source"]),
    ("Mental health", "Greater imbalance, more depression-related disability",
     f"HR {mental['individual_hr']:.2f} (individual measure) / {mental['work_unit_hr']:.2f} (work-unit measure)",
     "teal", mental["source"]),
    ("Mental health", "Long hours associated with depression",
     f"OR {hours['main_or']:.2f}; sensitivity analysis {hours['sensitivity_or']:.2f}, with interval including 1",
     "amber", hours["source"]),
    ("Mental health", "Insecurity–depression findings vary by group",
     f"Men: OR {insecurity['men_or']:.2f}; women: OR {insecurity['women_or']:.2f}, with interval including 1",
     "amber", insecurity["source"]),
    ("Life satisfaction", "Lower satisfaction; adjusted finding uncertain†",
     f"−{abs(security['adjusted_beta']):.2f} points (approx. 95% CI −{abs(security_ci[0]):.2f} to {security_ci[1]:.2f})",
     "amber", security["source"]),
]


In [ ]:
# 3. Draw the evidence diagram with Matplotlib; no image-generation service is used.
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
C={'ink':'#17303D','muted':'#607584','teal':'#007D72','amber':'#A36D16','gray':'#6D808D','line':'#D9E5E9','pale':'#EFF7F5'}
plt.rcParams.update({'font.family':'DejaVu Sans','text.color':C['ink'],'svg.fonttype':'none'})
f=plt.figure(figsize=(16,11),facecolor='white');a=f.add_axes([0,0,1,1]);a.set_xlim(0,1);a.set_ylim(0,1);a.axis('off')
def text(x,y,s,size=14,**kw):return a.text(x,y,s,fontsize=size,va='center',**kw)
def box(x,y,w,h,fill,edge):a.add_patch(FancyBboxPatch((x,y-h/2),w,h,boxstyle='round,pad=0,rounding_size=0.012',fc=fill,ec=edge,lw=1.2))
text(.04,.949,'How job quality relates to health and well-being',28,fontweight='bold')
text(.04,.903,'Seven investigated relationships • selected study findings, not causal effects',15,color=C['muted'])
text(.04,.850,'JOB-QUALITY FACTORS',12,fontweight='bold',color=C['muted'])
text(.30,.850,'WHAT THE STUDIES FOUND',12,fontweight='bold',color=C['muted'])
text(.795,.850,'OUTCOMES',12,fontweight='bold',color=C['muted'])
ys=[.785,.700,.595,.510,.405,.300,.215]
for y,label,h in [(.7425,'Wages and\nearnings',.151),(.5525,'Working\nconditions',.151),(.405,'Working hours',.072),(.2575,'Job security',.151)]:
 box(.04,y,.215,h,C['pale'],'#BEDBD5');text(.1475,y,label,20,fontweight='bold',ha='center',linespacing=1.35)
for y,(out,title,detail,col,url) in zip(ys,graph_rows):
 color=C[col]
 a.add_patch(FancyArrowPatch((.258,y),(.787,y),arrowstyle='-|>',mutation_scale=17,lw=2,color=color,linestyle=(0,(4,3)) if col=='gray' else '-'))
 # White label panel preserves the flow while keeping annotations readable.
 a.add_patch(FancyBboxPatch((.288,y-.032),.475,.065,boxstyle='round,pad=0.005,rounding_size=0.006',fc='white',ec='none',zorder=3))
 t=text(.30,y+.013,title,14.5,fontweight='bold',color=color,zorder=4);t.set_url(url)
 text(.30,y-.014,detail,11.4,color=C['muted'],zorder=4)
 box(.79,y,.17,.068,'#F0F5F9','#C7D4DE');text(.875,y,out,17,fontweight='bold',ha='center')
# Legend describes the displayed results, not evidence quality.
a.plot([.04,.96],[.159,.159],color=C['line'],lw=1)
for x,col,label in [(.04,'teal','Association observed'),(.31,'amber','Mixed / uncertain findings'),(.65,'gray','Interval not verified')]:
 a.plot([x,x+.025],[.134,.134],color=C[col],lw=3,ls='--' if col=='gray' else '-')
 text(x+.034,.134,label,12,color=C['muted'])
text(.04,.103,'RR = relative risk; HR = hazard ratio; OR = odds ratio. These measures and populations are not directly comparable.',11,color=C['muted'])
text(.04,.079,'† Life-satisfaction estimates are primary-paper supplements. * Physical-health estimate: abstract 1.32; Results text 1.31.',10.8,color=C['muted'])
text(.04,.055,'Sources: System Graph snapshot + checked papers. Salary is an SES proxy; physical-health outcomes include mortality and disability.',10.8,color=C['muted'])
text(.04,.029,'Colors describe selected findings, not evidence quality. Arrowheads identify the relationship studied. No pooling or attribute ranking.',10.8,color=C['muted'])

# 4. Export the graph for the presentation and show it in the notebook.
GRAPH_OUTPUT = Path("figures")
GRAPH_OUTPUT.mkdir(parents=True, exist_ok=True)
f.savefig(GRAPH_OUTPUT / "job_quality_relationships.png", dpi=200, facecolor="white")
f.savefig(GRAPH_OUTPUT / "job_quality_relationships.svg", facecolor="white")
plt.show()
print("Saved figures/job_quality_relationships.png and .svg")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
DATA = Path('data')
REQUIRED = ['study_screening', 'checked_estimates', 'data_quality_issues', 'shared_cohorts',
            'search_sensitivity', 'geographic_coverage', 'ilo_hours_2023', 'ilo_missing_2023',
            'wdi_context_2023', 'pooling_assessment', 'bias_review', 'expanded_associations',
            'expanded_pairs', 'remaining_study_queue']
missing = [name + '.csv' for name in REQUIRED if not (DATA / (name + '.csv')).exists()]
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the supplied CSVs in data/: ' + ', '.join(missing))
    print('Select all unzipped v02 CSV files. Your API key is not needed here.')
    uploaded = files.upload()
    DATA.mkdir(exist_ok=True)
    for name, content in uploaded.items():
        if name.endswith('.csv'):
            (DATA / Path(name).name).write_bytes(content)
    missing = [name + '.csv' for name in REQUIRED if not (DATA / (name + '.csv')).exists()]
    if missing:
        raise FileNotFoundError('Still missing: ' + ', '.join(missing))
tables = {name: pd.read_csv(DATA / (name + '.csv')).fillna('') for name in REQUIRED}
screen = tables['study_screening']; estimates = tables['checked_estimates']
print(f"Loaded {len(screen)} screened publications and {len(estimates)} extracted estimates.")


## 1. Screening decisions and exclusions
“Retain for full text” is a preliminary decision, not final eligibility. Reviews, reverse-direction analyses, clinical disorders and distinct comparators are kept in separate categories. A paper's presence in the Graph does not validate its exposure/outcome match.


In [ ]:
display(screen.groupby('screening_decision').size().rename('publications').to_frame())
display(screen.loc[screen.screening_decision != 'retain_for_fulltext',
    ['pmid','title','screening_decision','decision_reason','notes','source_url']])
print('Review stages:')
display(screen.groupby('review_stage').size().rename('publications').to_frame())


## 2. Shared cohorts
Ten potential overlap clusters require checking waves, participants and outcomes. Distinct publication IDs are not necessarily independent studies. Cohort membership alone does not prove that samples are identical.


In [ ]:
display(tables['shared_cohorts'])


## 3. Sensitivity to expanded search terms
Across all categories the original search found **614 distinct associations / 418 publication IDs**; expansion finds **1,004 / 653**. Counts in individual cells overlap and must not be added as independent evidence. Physical-health expansion includes disease-specific endpoints; quality of life is broader than life satisfaction. Informality still has sparse matches. A zero means no match to this vocabulary in this snapshot, not no research.


In [ ]:
sensitivity = tables['search_sensitivity']
display(sensitivity)
pivot = sensitivity.pivot(index='attribute', columns='outcome', values='candidate_associations_added')
fig, ax = plt.subplots(figsize=(11,6))
im = ax.imshow(pivot.to_numpy(dtype=float), cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha='right')
ax.set_yticks(range(len(pivot.index)), pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j,i,str(int(pivot.iloc[i,j])),ha='center',va='center',
                color='white' if pivot.iloc[i,j] > 100 else 'black')
ax.set_title('Additional candidate associations after vocabulary expansion')
fig.colorbar(im, ax=ax, label='Overlapping discovery counts')
fig.tight_layout(); plt.show()


### Inspect a candidate group
Change the two strings below. Unscreened new candidates remain explicitly labeled; do not treat them as included studies.


In [ ]:
ATTRIBUTE = 'Job security'
OUTCOME = 'Depression / anxiety'
pairs = tables['expanded_pairs']
selected = pairs[(pairs.attribute == ATTRIBUTE) & (pairs.outcome == OUTCOME)]
selected = selected.merge(screen[['study_id','pmid','screening_decision','review_stage']], on='study_id', how='left')
selected['screening_decision'] = selected.screening_decision.fillna('not screened in priority pass')
display(selected)
print('Expanded publications outside the original priority screen:', len(tables['remaining_study_queue']))


## 4. Checked estimates, including uncertain results
The ledger preserves the paper's metric, comparator, adjustment, subgroup and source location. “Full-text table checked” describes targeted extraction, not a full risk-of-bias review. Graph matches are numeric matches only; direction and contrast come from paper checking.

In the German S-MGA cohort, male and female intervals differ in precision. This does not demonstrate a sex interaction or rank wages against insecurity. The UK Biobank shift/nonshift and night/nonnight estimates ask different questions. Canadian rotation estimates and the male-production-worker absence estimate remain abstract-only.


In [ ]:
display(estimates[['estimate_id','pmid','exposure','comparator','outcome','measure','estimate',
                   'ci_lower','ci_upper','subgroup','validation_stage','hold_reason','source_url']])
german = estimates[estimates.pmid.astype(str) == '35328860'].copy()
fig, ax = plt.subplots(figsize=(10,4))
y = np.arange(len(german))
v = german.estimate.astype(float).to_numpy()
lo = german.ci_lower.astype(float).to_numpy(); hi = german.ci_upper.astype(float).to_numpy()
ax.errorbar(v,y,xerr=np.vstack([v-lo,hi-v]),fmt='o',capsize=4,color='#185a70')
ax.axvline(1,color='gray',linestyle='--')
ax.set_yticks(y, german.subgroup + ': ' + german.exposure)
ax.invert_yaxis();ax.set_xscale('log');ax.set_xlabel('Adjusted odds ratio (95% CI), logarithmic axis')
ax.set_title('One German cohort: separate exposure models, no pooled estimate')
fig.tight_layout();plt.show()
print('Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC8949946/ Tables 5–6')


### Data discrepancies
Nine recorded issues include false population size, survey years, comparator coding, metric ambiguity and abstract/table discrepancies. Holds remain visible even when a plausible table value is available. No correction is inferred for an estimate outside its stated interval.


In [ ]:
display(tables['data_quality_issues'])


## 5. Study populations and income vintage
Countries come from sampled populations, not author affiliations. Of the 75 retained priority candidates, **63 have a resolved country and 12 remain unresolved**. Taiwan's income group remains unassigned in this World Bank lookup. Classification uses **FY2027**, retrieved 14 November 2026, not the income group during each study's data collection. These are sample coverage counts, not country health outcomes.


In [ ]:
display(tables['geographic_coverage'])
display(screen.loc[screen.study_country_iso3 == '', ['pmid','title','screening_decision','country_status']])


## 6. Matched country context, with gaps preserved
ILOSTAT: 2023, both sexes, average weekly hours actually worked, main job. National age coverage and survey methods still differ; consult decoded notes. Available for **15 of 22 requested settings**. Pakistan was carried forward as an original project comparator; remaining settings come from screened populations.

WDI vulnerable employment: own-account plus contributing family employment as a percentage of employment; modeled ILO estimates. It is not synonymous with perceived insecurity or informal employment. Four country values were retrieved; requests for additional countries failed. Retrieval failure is distinct from missing source data. Only Brazil, the UK and the US currently have both context measures in this package.

The common year aligns context measures with one another, **not with every historical study period**. No health inference is made from these tables. GLD microdata remain unintegrated pending an accessible survey file and documented weights, version and definitions.


In [ ]:
ilo = tables['ilo_hours_2023']; wdi = tables['wdi_context_2023']
display(ilo[['ref_area','country','time','obs_value','unit','source_label','note_indicator_label','note_source_label']])
display(tables['ilo_missing_2023'])
display(wdi[['iso3','country','year','value','status']])
matched = ilo[['ref_area','obs_value']].merge(wdi.loc[wdi.status == 'retrieved', ['iso3','value']],
    left_on='ref_area', right_on='iso3').rename(columns={'obs_value':'weekly_hours','value':'vulnerable_employment_pct'})
display(matched[['iso3','weekly_hours','vulnerable_employment_pct']])


## 7. Bias and synthesis decisions
No group is approved for pooling. Exposure definitions, thresholds, outcomes, designs, effect measures and sampling independence must align first. A null-compatible interval is not proof of no association; retaining it prevents selective reporting.

The bias table lists outstanding domain questions. It is **not** a formal ROBINS-E/ROBINS-I or GRADE assessment. No overall bias grades or policy effect estimates have been assigned.


In [ ]:
display(tables['pooling_assessment'])
display(tables['bias_review'])


## What remains
1. Independently verify extraction, retrieve unresolved full texts and finish eligibility decisions for the 75 retained priority candidates.
2. Resolve cohort waves and participants; screen the 566 expanded-search publications outside the original priority set.
3. Complete population geography and study-period income classifications if historical comparisons are needed.
4. Retry missing World Bank context, align context to study years where relevant, and obtain an authorized GLD survey.
5. Complete formal domain-level bias appraisal; assess pooling only within compatible independent groups.

### Live API status
This notebook's findings use the public Dataverse snapshot, Europe PMC papers, ILOSTAT and World Bank data. No authenticated System API run was made here. Adding a Colab secret does not automatically execute a request. The separately supplied `system_client.py` remains available for a deliberate live-API extension; its output must be checked against papers too.

### Sources and audit
The package includes `SYNTHESIS.md`, the plain CSV ledgers, `SOURCE_MANIFEST.json`, `discover.py`, `expand_search.py` and audit rules. Source links and locations are recorded per paper/estimate. Public raw abstracts/full texts were used for review but are not republished in the package.
